# Infra-FM: STAC Imagery Fetch

Fetches Sentinel-2 + Sentinel-1 tiles for infrastructure assets across multiple regions.

**Regions in this notebook:** australia-oceania, africa, south-america

**Before running:**
1. Upload your deduped parquet files to Google Drive:
   - `infra_fm/pipeline/africa_deduped_assets_substations.parquet`
   - `infra_fm/pipeline/australia-oceania_deduped_assets_substations.parquet`
   - `infra_fm/pipeline/south-america_deduped_assets_substations.parquet`
2. Upload your curation code zip to Drive: `infra_fm/code/infra_fm_curation.zip`
3. Run cells top to bottom — checkpoints save to Drive so you can resume if disconnected.

## 1. Mount Google Drive

In [21]:
from google.colab import drive
drive.mount('/content/drive')

# Verify your files are visible
import os
DRIVE_ROOT = '/content/drive/MyDrive/infra_fm'
print('Drive root contents:')
for f in sorted(os.listdir(DRIVE_ROOT)):
    print(' ', f)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive root contents:
  checkpoints
  code
  datasets
  pipeline


## 2. Install dependencies

In [19]:
!pip install -q pystac-client planetary-computer rasterio scipy opencv-python-headless

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.5/208.5 kB 7.5 MB/s eta 0:00:00


## 3. Set up curation code

In [22]:
import subprocess, shutil, sys
from pathlib import Path

CODE_ZIP  = f'{DRIVE_ROOT}/code/infra_fm_curation.zip'
CODE_DEST = '/content/infra_fm/infra_fm_code_only/curation'

if Path(CODE_ZIP).exists():
    shutil.unpack_archive(CODE_ZIP, CODE_DEST)
    print(f'Extracted code to {CODE_DEST}')
else:
    raise FileNotFoundError(
        f'Code zip not found at {CODE_ZIP}.\n'
        'Zip your local infra_fm/ folder and upload to Drive.'
    )

# Add to Python path so imports work
sys.path.insert(0, CODE_DEST)
print('Python path updated')
print('Contents:', sorted(os.listdir(CODE_DEST)))

Extracted code to /content/infra_fm/infra_fm_code_only/curation
Python path updated
Contents: ['infra_fm_code_only\\.gitignore', 'infra_fm_code_only\\README.md', 'infra_fm_code_only\\additional_info\\ONTOLOGY.md', 'infra_fm_code_only\\additional_info\\PROJECT_SUMMARY.md', 'infra_fm_code_only\\caadd_asset_table_code_test.py', 'infra_fm_code_only\\curation\\__pycache__\\', 'infra_fm_code_only\\curation\\central_america_preview.png', 'infra_fm_code_only\\curation\\data\\', 'infra_fm_code_only\\curation\\data\\checkpoints\\', 'infra_fm_code_only\\curation\\dataset.py', 'infra_fm_code_only\\curation\\deduplication.py', 'infra_fm_code_only\\curation\\extract_substations_all.py', 'infra_fm_code_only\\curation\\gee_imagery.py', 'infra_fm_code_only\\curation\\helpers\\__pycache__\\', 'infra_fm_code_only\\curation\\helpers\\compare_schedulers.py', 'infra_fm_code_only\\curation\\helpers\\csv_to_parquet.py', 'infra_fm_code_only\\curation\\helpers\\log_power_only_run.py', 'infra_fm_code_only\\curat

## 4. Configuration — edit this cell before running

In [23]:
# --- Paths ---
PIPELINE_DIR   = f'{DRIVE_ROOT}/pipeline'       # where your deduped parquets live
DATASETS_DIR   = f'{DRIVE_ROOT}/datasets'       # where curated datasets will be written
CHECKPOINT_DIR = f'{DRIVE_ROOT}/checkpoints'    # fetch checkpoints — survives disconnects

os.makedirs(DATASETS_DIR,   exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# --- Fetch settings ---
MODALITIES   = ['sentinel2_ms', 'sentinel1']   # landsat_thermal dropped for speed
BUFFER_M     = 300
MAX_WORKERS  = 64    # Colab has good network — push concurrency higher than local
START_WORKERS = 32

# --- Regions to process (in order — smallest first) ---
REGIONS = [
    'australia-oceania',   # ~5,583 assets  ~3-4h
    'africa',              # ~11,044 assets ~6-8h
    'south-america',       # ~17,899 assets ~10-12h
]

# Skip regions that already have a _SUCCESS file
SKIP_DONE = True

print('Configuration:')
print(f'  Modalities:    {MODALITIES}')
print(f'  Buffer:        {BUFFER_M}m')
print(f'  Workers:       {START_WORKERS} start / {MAX_WORKERS} max')
print(f'  Regions:       {REGIONS}')
print(f'  Datasets dir:  {DATASETS_DIR}')
print(f'  Checkpoints:   {CHECKPOINT_DIR}')

Configuration:
  Modalities:    ['sentinel2_ms', 'sentinel1']
  Buffer:        300m
  Workers:       32 start / 64 max
  Regions:       ['australia-oceania', 'africa', 'south-america']
  Datasets dir:  /content/drive/MyDrive/infra_fm/datasets
  Checkpoints:   /content/drive/MyDrive/infra_fm/checkpoints


## 5. Verify deduped parquets are accessible

In [24]:
import pandas as pd

for region in REGIONS:
    path = f'{PIPELINE_DIR}/{region}_deduped_assets_substations.parquet'
    if Path(path).exists():
        df = pd.read_parquet(path)
        print(f'{region:25s} {len(df):>6,} assets — OK')
    else:
        print(f'{region:25s} MISSING — {path}')

australia-oceania          5,583 assets — OK
africa                    11,044 assets — OK
south-america             17,899 assets — OK


## 6. Run STAC fetch pipeline — all regions

This cell runs the full pipeline (fetch → QC → triage → assemble) for each region.
Checkpoints are saved to Drive every 200 tiles — if the session disconnects, re-run
this cell and it will resume from the last checkpoint automatically.

In [ ]:
import sys, os

# Point to the curation folder where stac_imagery.py actually lives
CURATION_DIR = '/content'
sys.path.insert(0, CURATION_DIR)
os.chdir(CURATION_DIR)

import json
from datetime import datetime

from stac_imagery import STACImageryFetcher
from qc import QualityChecker
from triage import RuleBasedTriager
from dataset import DatasetAssembler
from io_utils import load_asset_table


def is_done(region):
    success = Path(f'{DATASETS_DIR}/dataset_{region}_stac_v1/_SUCCESS')
    return success.exists()


def run_region(region):
    print('\n' + '=' * 60)
    print(f'Region: {region}  ({datetime.now().strftime("%H:%M:%S")})')
    print('=' * 60)

    if SKIP_DONE and is_done(region):
        print(f'  Already complete — skipping.')
        return

    # Load deduped assets
    parquet = f'{PIPELINE_DIR}/{region}_deduped_assets_substations.parquet'
    df = load_asset_table(parquet)
    print(f'  Loaded {len(df):,} assets')

    output_dir      = f'{DATASETS_DIR}/dataset_{region}_stac_v1'
    checkpoint_path = f'{CHECKPOINT_DIR}/{region}_fetch.pkl'
    os.makedirs(output_dir, exist_ok=True)

    # --- Step 1: Fetch imagery ---
    print(f'  [1/4] Fetching imagery...')
    fetcher = STACImageryFetcher(
        buffer_m             = BUFFER_M,
        modalities           = MODALITIES,
        temporal_stack       = False,
        checkpoint_path      = checkpoint_path,
        checkpoint_every     = 200,
        adaptive_concurrency = True,
        start_workers        = START_WORKERS,
        max_workers          = MAX_WORKERS,
    )
    tiles = fetcher.fetch_all(df)
    n_ok  = sum(1 for t in tiles if t.status == 'ok')
    print(f'  Fetched: {n_ok} ok / {len(tiles) - n_ok} failed')

    # --- Step 2: QC ---
    print(f'  [2/4] Quality control...')
    checker    = QualityChecker(min_valid_ratio=0.80)
    qc_results = checker.check_all(tiles, max_workers=4)
    clean      = checker.filter_ok(qc_results)
    print(f'  QC passed: {len(clean)} / {len(tiles)}')

    # --- Step 3: Triage ---
    print(f'  [3/4] Confidence triage...')
    triager        = RuleBasedTriager(contradiction_threshold=3, low_threshold=4)
    triage_results = triager.triage_all(clean, max_workers=4)
    accepted       = triager.filter_accepted(triage_results)
    print(f'  Accepted: {len(accepted)}')

    # --- Step 4: Assemble ---
    print(f'  [4/4] Assembling dataset -> {output_dir}')
    assembler = DatasetAssembler(output_dir)
    summary   = assembler.assemble(accepted, triage_results)

    # Write _SUCCESS
    success_path = Path(output_dir) / '_SUCCESS'
    with open(success_path, 'w') as f:
        json.dump({
            'completed_at':   datetime.utcnow().isoformat() + 'Z',
            'region':         region,
            'n_dataset_tiles': len(summary),
            'modalities':     MODALITIES,
        }, f, indent=2)

    print(f'  Done. {len(summary)} tiles assembled.')
    return len(summary)


# Run all regions
results = {}
for region in REGIONS:
    try:
        n = run_region(region)
        results[region] = n or 'skipped'
    except Exception as e:
        print(f'ERROR in {region}: {e}')
        results[region] = f'error: {e}'

print('\n' + '=' * 40)
print('SUMMARY')
print('=' * 40)
for region, result in results.items():
    print(f'  {region:25s} {result}')


Region: australia-oceania  (17:46:12)
  Loaded 5,583 assets
  [1/4] Fetching imagery...
STACImageryFetcher: modalities=['sentinel2_ms', 'sentinel1'], n_bands=9, temporal_stack=False, buffer_m=300, workers=32 (adaptive)
  STAC fetch: 5583 assets pending (0 already checkpointed)
  [10/5583] (0%) ok=1 fail=9 workers=32
  [20/5583] (0%) ok=8 fail=12 workers=32
  [concurrency] workers=24 | throughput=0.1 tiles/s | fail_rate=60.0% | ↓ back off (high fail rate)
  [30/5583] (1%) ok=17 fail=13 workers=24
  [40/5583] (1%) ok=22 fail=18 workers=24
  [concurrency] workers=16 | throughput=0.4 tiles/s | fail_rate=30.0% | ↓ back off (high fail rate)
  [50/5583] (1%) ok=26 fail=24 workers=16
  [60/5583] (1%) ok=29 fail=31 workers=16
  [concurrency] workers=8 | throughput=0.2 tiles/s | fail_rate=65.0% | ↓ back off (high fail rate)
  [70/5583] (1%) ok=32 fail=38 workers=8
  [80/5583] (1%) ok=34 fail=46 workers=8
  [concurrency] workers=8 | throughput=0.1 tiles/s | fail_rate=75.0% | ↓ back off (high fai

/content/stac_imagery.py:241: RuntimeWarning: invalid value encountered in log10
  arr = np.where(arr > 0, 10 * np.log10(arr + 1e-10), vmin)


  [140/5583] (3%) ok=71 fail=69 workers=8
  [concurrency] workers=8 | throughput=0.4 tiles/s | fail_rate=45.0% | ↓ back off (high fail rate)
  [150/5583] (3%) ok=79 fail=71 workers=8
  [160/5583] (3%) ok=86 fail=74 workers=8
  [concurrency] workers=8 | throughput=0.4 tiles/s | fail_rate=25.0% | ↓ back off (high fail rate)


KeyboardInterrupt: 

In [ ]:
import os, sys

# See what's actually in the extracted location
print("Contents of /content/infra_fm:")
for item in sorted(os.listdir('/content/infra_fm')):
    print(' ', item)

print("\nLooking for stac_imagery.py:")
for root, dirs, files in os.walk('/content'):
    for f in files:
        if 'stac_imagery' in f:
            print(' ', os.path.join(root, f))

## 7. Verify completed datasets

In [ ]:
import json

print('Dataset status:')
for region in REGIONS:
    success_path = Path(f'{DATASETS_DIR}/dataset_{region}_stac_v1/_SUCCESS')
    if success_path.exists():
        meta = json.loads(success_path.read_text())
        print(f'  {region:25s} DONE — {meta["n_dataset_tiles"]:,} tiles')
    else:
        # Check if partially complete via checkpoint
        ckpt = Path(f'{CHECKPOINT_DIR}/{region}_fetch.pkl')
        if ckpt.exists():
            import pickle
            data = pickle.load(open(ckpt, 'rb'))
            n_done = len(data.get('completed_ids', []))
            print(f'  {region:25s} IN PROGRESS — {n_done:,} tiles fetched so far')
        else:
            print(f'  {region:25s} NOT STARTED')

## 8. Download completed datasets to Drive (already done — they write there directly)

Your datasets are written directly to `My Drive/infra_fm/datasets/`.
To use them locally:
1. Download each `dataset_<region>_stac_v1/` folder from Drive to your local `data/curated_datasets/`
2. Or run pretraining directly from Colab (see pretraining notebook)

In [18]:
import os

print("Contents of /content/ (top level):")
for item in sorted(os.listdir('/content')):
    if not item.startswith('.') and item not in ['drive', 'sample_data']:
        print(f"  {item}")

print("\nContents of /content/legacy/ (if exists):")
if os.path.exists('/content/legacy'):
    for item in sorted(os.listdir('/content/legacy')):
        print(f"  {item}")
else:
    print("  (does not exist)")

print("\nContents of /content/helpers/ (if exists):")
if os.path.exists('/content/helpers'):
    for item in sorted(os.listdir('/content/helpers')):
        print(f"  {item}")
else:
    print("  (does not exist)")

print("\nContents of /content/utils/ (if exists):")
if os.path.exists('/content/utils'):
    for item in sorted(os.listdir('/content/utils')):
        print(f"  {item}")
else:
    print("  (does not exist)")

print("\nKey files present:")
for f in ['stac_imagery.py', 'qc.py', 'triage.py', 'dataset.py', 'sources.py']:
    print(f"  {f}: {os.path.exists(f'/content/{f}')}")

Contents of /content/ (top level):
  __pycache__
  dataset.py
  helpers
  infra_fm
  legacy
  pipeline.py
  qc.py
  sources.py
  stac_imagery.py
  triage.py
  utils

Contents of /content/legacy/ (if exists):
  __pycache__
  imagery.py

Contents of /content/helpers/ (if exists):
  __pycache__
  tile_types.py

Contents of /content/utils/ (if exists):
  io_utils.py
  timing_log_utils.py

Key files present:
  stac_imagery.py: True
  qc.py: True
  triage.py: True
  dataset.py: True
  sources.py: True
